<a href="https://colab.research.google.com/github/engineerchacon/Procesamiento-de-Lenguaje-Natural/blob/main/NER_Armando_Chac%C3%B3n_Terrazas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **NER-Armando Chacón Terrazas**

Aplica CRF para el Reconocimiento de Entidades Nombradas al conjunto de datos proporcionado.


Instrucciones:

1.	Carga el conjunto de datos y revisa su contenido.
2.	Preprocesa los datos para generar las frases tal como las vistas en las diapositivas 19 y 20 de la presentación:
[
 ('Interesante', 'ADJ', 'O'),
 ('será', 'VLfin', 'O'),
 ('conocer', 'VLinf', 'O'),
 ('Pedro', 'NP', 'B-PER'),
 ('Kumamoto', 'NC', 'E-PER')
]

Para lo anterior, considera el siguiente código y, en caso necesario, haz las modificaciones necesarias (no he probado este código):




def agrupar_oraciones(df):
    oraciones = []

    grouped = df.groupby("Sentence #")

    for _, grupo in grouped:
        sentence = [(w, p, t) for w, p, t in zip(grupo["Word"], grupo["Pos"], grupo["Tag"])]
        oraciones.append(sentence)

    return oraciones


sentences = agrupar_oraciones(df)
print(sentences[0])

3.	Genera el conjunto train con el 80% de las frases formadas y el 20% para el conjunto test.
4.	Aplica las métricas para evaluar el desempeño del modelo.
5.	Muestra la matriz de confusión.
6.	Identifica y menciona cuál fue la etiqueta NER que consideres fue en la que CRF tuvo el más bajo desempeño.


**CRF (Conditional Random Fields)** o Campos Aleatorios Condicionales es un modelo estadístico utilizado para el etiquetado de secuencias. En el contexto del Procesamiento de Lenguaje Natural (PLN):

**Función:** Se utiliza para determinar si un token (palabra) representa una entidad específica basándose en el contexto.

**Aprendizaje:** A diferencia de los modelos basados en reglas, los CRF aprenden patrones a partir de un corpus anotado (datos etiquetados previamente).

**Contexto:** El modelo extrae características no solo de la palabra actual, sino también de su "contexto izquierdo" (palabra anterior) y "contexto derecho" (palabra siguiente) para mejorar la precisión de la etiqueta asignada.

# **1: Preparación del Entorno y Carga de Datos**

In [4]:
# Conexión con Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Instalación de la librería para el modelo CRF
!pip install sklearn-crfsuite
import pandas as pd

# Se define la ruta y se lee el archivo
ruta_archivo = '/content/drive/MyDrive/UACJ/Procesamiento de lenguaje natural/split1.mx-news.txt'
df = pd.read_csv(ruta_archivo, sep='\t', quoting=3).dropna(how='all')

print("Celda 1 ejecutada: Datos cargados correctamente.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Celda 1 ejecutada: Datos cargados correctamente.


# **2: Preprocesamiento de Oraciones**

In [5]:
# Se integra la función
def agrupar_oraciones(df):
    oraciones = []
    # Se agrupan todas las filas que comparten el mismo número de "Sentence #"
    grouped = df.groupby("Sentence #")

    for _, grupo in grouped:
        # Se extrae y empaqueta en tríos (Palabra, Pos, Tag)
        sentence = [(w, p, t) for w, p, t in zip(grupo["Word"], grupo["Pos"], grupo["Tag"])]
        oraciones.append(sentence)

    return oraciones

# Se manda a llamar a la función
frases_listas = agrupar_oraciones(df)

print(f"Celda 2 ejecutada: Se han generado {len(frases_listas)} frases con el formato correcto.")

Celda 2 ejecutada: Se han generado 1295 frases con el formato correcto.


# **3: Extracción de Características**

In [6]:
# Función para extraer las características (pistas) de cada palabra
def palabra_a_caracteristicas(oracion, i):
    palabra = oracion[i][0]
    pos = oracion[i][1]

    caracteristicas = {
        'palabra.lower()': palabra.lower(),
        'palabra.isupper()': palabra.isupper(),
        'palabra.istitle()': palabra.istitle(),
        'palabra.isdigit()': palabra.isdigit(),
        'pos': pos
    }

    # Contexto anterior
    if i > 0:
        caracteristicas.update({
            '-1:palabra.lower()': oracion[i-1][0].lower(),
            '-1:palabra.istitle()': oracion[i-1][0].istitle(),
            '-1:pos': oracion[i-1][1],
        })
    else:
        caracteristicas['BOS'] = True # Inicio de oración

    # Contexto siguiente
    if i < len(oracion) - 1:
        caracteristicas.update({
            '+1:palabra.lower()': oracion[i+1][0].lower(),
            '+1:palabra.istitle()': oracion[i+1][0].istitle(),
            '+1:pos': oracion[i+1][1],
        })
    else:
        caracteristicas['EOS'] = True # Fin de oración

    return caracteristicas

# Se convierten las frases en variables X (características) e Y (etiquetas)
X_datos = [[palabra_a_caracteristicas(s, i) for i in range(len(s))] for s in frases_listas]
y_datos = [[etiqueta for palabra, pos, etiqueta in s] for s in frases_listas]

print("Variables X_datos e y_datos listas para usar.")

Variables X_datos e y_datos listas para usar.


# **4: División 80/20 y Entrenamiento**

In [7]:
from sklearn.model_selection import train_test_split
import sklearn_crfsuite

# Separación de los datos en Entrenamiento (80%) y Prueba (20%)
X_entrenamiento, X_prueba, y_entrenamiento, y_prueba = train_test_split(
    X_datos,
    y_datos,
    test_size=0.20,
    random_state=42
)

# Configuración del Modelo CRF
modelo_crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.1,
    c2=0.1,
    max_iterations=100,
    all_possible_transitions=True
)

# Entrenamiento del modelo
print("Iniciando el entrenamiento del modelo CRF")
modelo_crf.fit(X_entrenamiento, y_entrenamiento)

print("El modelo ha sido entrenado exitosamente")

Iniciando el entrenamiento del modelo CRF
El modelo ha sido entrenado exitosamente


# **5: Evaluación Final (Métricas, Matriz y Peor Desempeño)**

In [8]:
from sklearn_crfsuite import metrics
import pandas as pd

# Se generan las predicciones con el 20% de prueba
y_prediccion = modelo_crf.predict(X_prueba)

# Se omiten las etiquetas 'O' para una evaluación más limpia de las entidades reales
etiquetas = list(modelo_crf.classes_)
etiquetas.remove('O')

print("MÉTRICAS DE DESEMPEÑO")
reporte = metrics.flat_classification_report(y_prueba, y_prediccion, labels=etiquetas, digits=3)
print(reporte)

print("\nMATRIZ DE CONFUSIÓN")
# Se aplanan las listas para poder generar la matriz cruzada
y_prueba_plano = [etiqueta for oracion in y_prueba for etiqueta in oracion]
y_pred_plano = [etiqueta for oracion in y_prediccion for etiqueta in oracion]

matriz = pd.crosstab(pd.Series(y_prueba_plano, name='Real'),
                     pd.Series(y_pred_plano, name='Predicción'),
                     margins=True)
print(matriz)

print("\nETIQUETA CON EL MÁS BAJO DESEMPEÑO")
# Se convierte el reporte a diccionario para analizarlo mediante código
reporte_dict = metrics.flat_classification_report(y_prueba, y_prediccion, labels=etiquetas, output_dict=True)

# Se busca la etiqueta con el menor F1-Score (omitiendo los promedios 'avg')
peor_etiqueta = min(
    [tag for tag in reporte_dict.keys() if "avg" not in tag],
    key=lambda x: reporte_dict[x]['f1-score']
)
peor_f1 = reporte_dict[peor_etiqueta]['f1-score']

print(f"La etiqueta NER con el más bajo desempeño fue: '{peor_etiqueta}' con un F1-Score de {peor_f1:.3f}")

MÉTRICAS DE DESEMPEÑO
              precision    recall  f1-score   support

       B-MNY      1.000     0.833     0.909        12
       I-MNY      1.000     1.000     1.000        22
       E-MNY      1.000     0.833     0.909        12
       S-PER      1.000     0.717     0.835        46
       B-TIT      0.854     0.651     0.739        63
       I-TIT      0.703     0.617     0.657       115
       E-TIT      0.750     0.571     0.649        63
       B-EVT      0.786     0.647     0.710        17
       E-EVT      0.786     0.647     0.710        17
       S-GPE      0.823     0.855     0.839        76
       S-DAT      0.905     0.950     0.927        80
       S-TIT      0.826     0.500     0.623        38
       S-ORG      0.793     0.479     0.597        48
       I-EVT      0.682     0.789     0.732        19
       B-DOC      0.800     0.235     0.364        17
       I-DOC      1.000     0.444     0.615        45
       E-DOC      1.000     0.294     0.455        17
     

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/me

**Conclusión del Modelo NER (CRF)**

El algoritmo CRF demostró ser altamente capaz para este conjunto de datos, logrando un F1-Score de 81.1% y una precisión de 87.2%. Estos números confirman que, a pesar de no usar arquitecturas modernas como redes neuronales, la estrategia de extraer pistas simples del texto (como el uso de mayúsculas, el contexto inmediato y el tipo de palabra) fue suficiente para que el modelo encontrara patrones sólidos.

**Puntos Fuertes (Aciertos del Modelo)**

Precisión casi perfecta en formatos fijos: Las fechas y el dinero superaron el 90% de éxito. Al usar constantemente números y símbolos, el modelo encontró muy fácil aprenderse sus reglas.

Reconocimiento de Personas y Organizaciones: Identificó nombres propios y empresas de manera muy efectiva (entre 78% y 89%), aprovechando inteligentemente las letras iniciales mayúsculas y las palabras vecinas.

Predicciones seguras: El modelo demostró ser "conservador". Al tener una precisión más alta que su recall, significa que cuando etiqueta una palabra, es casi seguro que esté en lo correcto; prefiere ignorar palabras dudosas antes que cometer equivocaciones.

**Puntos Débiles (Áreas de Mejora)**

Confusión con nombres complejos: Los Documentos y los Productos representaron el mayor desafío (30% a 60% de acierto). Como sus nombres suelen ser frases largas o descriptivas, el modelo tiende a confundirlos con texto común y corriente.

**La etiqueta NER en la que el modelo CRF tuvo el más bajo desempeño fue S-EVT (Eventos individuales), la cual obtuvo un F1-Score de 0.000, al igual que otras etiquetas minoritarias como S-LOC, S-PRO e I-AGE.**

Este resultado no se debe a un error de programación ni a una falla en el algoritmo CRF, sino a un problema clásico en ciencia de datos conocido como desbalance de clases o falta de soporte (Low Support).

Si se observa la columna "support" en la tabla de métricas (que indica cuántas veces apareció esa etiqueta en el examen final), la etiqueta S-EVT tiene un valor de 1. Al tener un solo ejemplo disponible (o cero en los otros casos), el modelo matemáticamente no tuvo la cantidad de datos suficientes durante el entrenamiento para aprender a reconocer los patrones gramaticales y contextuales de esa entidad. Al no poder generalizar, el modelo es incapaz de predecirla correctamente, resultando en una precisión y recall de cero.